# Практика 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.1.0


## Два входных файла

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## Профиль входа

In [3]:
print(export.isna().sum())
print(export["action_taken"].value_counts(dropna=False))

source_row         0
activity_year      0
lei                0
county_code        0
action_taken       0
loan_amount        0
income           342
dtype: int64
action_taken
1     5957
3      691
2      183
88       1
Name: count, dtype: Int64


## A. Проверки входа

In [4]:
print(export["source_row"].duplicated().sum())
print(export.loc[~export["action_taken"].isin(["1", "2", "3"]), ["source_row", "action_taken"]])
print(export["county_code"].value_counts(dropna=False))

2
  source_row action_taken
1          3           88
county_code
11001      6831
 11001        1
Name: count, dtype: Int64


## Преобразование суммы

In [5]:
work = export.copy(deep=True)
work["county_code"] = work["county_code"].str.strip()
for field in ["source_row", "loan_amount", "action_taken"]:
    numbers = pd.to_numeric(work[field], errors="coerce")
    failed = work[field].notna() & numbers.isna()
    print(field, int(failed.sum()))
    print(work.loc[failed, ["source_row", field]])
    work[field] = numbers

source_row 0
Empty DataFrame
Columns: [source_row, source_row]
Index: []
loan_amount 1
   source_row loan_amount
0           2        oops
action_taken 0
Empty DataFrame
Columns: [source_row, action_taken]
Index: []


## B. Ошибки преобразования дохода

In [6]:
income_numbers = pd.to_numeric(export["income"], errors="coerce")
income_parse_error = export["income"].notna() & income_numbers.isna()
print(export.loc[income_parse_error, ["source_row", "income"]])
print("Новых ошибок:", income_parse_error.sum())
print("Исходных пропусков:", export["income"].isna().sum())
print("Пропусков после преобразования:", income_numbers.isna().sum())
assert_frame_equal(export, raw_copy)

  source_row income
2          5    BAD
Новых ошибок: 1
Исходных пропусков: 342
Пропусков после преобразования: 343


## Проверка повторной доставки

In [7]:
known_key = export["source_row"].notna()
repeated = export.loc[known_key].drop_duplicates()
assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
replay_mask = known_key & export["source_row"].duplicated()
unique = work.loc[~replay_mask].copy()
print(len(export), len(unique), int(replay_mask.sum()))

6832 6830 2


## C. Строки с нарушениями обязательных правил

In [8]:
unique = export.drop_duplicates(keep="first").copy()
loan_num = pd.to_numeric(unique["loan_amount"], errors="coerce")
action_num = pd.to_numeric(unique["action_taken"], errors="coerce")
bad_core = (loan_num.isna()) | (loan_num <= 0) | (~action_num.isin([1, 2, 3]))
quarantine_part = unique.loc[bad_core].copy()
clean_part = unique.loc[~bad_core].copy()
print("Спорные строки:")
print(quarantine_part[["source_row", "loan_amount", "action_taken"]])
n_input = len(export)
n_repeats = export["source_row"].duplicated().sum()
print(f"Баланс: {n_input} ")

Спорные строки:
  source_row loan_amount action_taken
0          2        oops            1
1          3    585000.0           88
Баланс: 6832 


## Функция по разобранным шагам

In [9]:
def clean_export(source):
    out = source.copy(deep=True)
    known_key = out["source_row"].notna()
    repeated = out.loc[known_key].drop_duplicates()
    assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
    replay_mask = known_key & out["source_row"].duplicated()
    replay_n = int(replay_mask.sum())
    out = out.loc[~replay_mask].copy()
    out["county_code"] = out["county_code"].str.strip()
    parse_counts = {}
    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(out[field], errors="coerce")
        parse_counts[field] = int((out[field].notna() & numbers.isna()).sum())
        out[field] = numbers
    reason = pd.Series("", index=out.index, dtype="string")
    reason.loc[out["source_row"].isna()] += "unknown_source_row;"
    bad_amount = out["loan_amount"].isna() | out["loan_amount"].le(0)
    reason.loc[bad_amount] += "invalid_loan_amount;"
    reason.loc[~out["action_taken"].isin([1, 2, 3])] += "invalid_action_taken;"
    bad_county = ~out["county_code"].eq("11001").fillna(False)
    reason.loc[bad_county] += "invalid_county;"
    invalid = reason.ne("")
    clean = out.loc[~invalid].copy()
    quarantine = source.loc[out.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]
    audit = {
        "input": len(source), "replays": replay_n,
        "clean": len(clean), "quarantine": len(quarantine),
        "parse_errors": parse_counts,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit

## Результат функции

In [10]:
result, rejected, audit = clean_export(export)
print(audit)
rejected

{'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': {'source_row': 0, 'loan_amount': 1, 'income': 1, 'action_taken': 0}}


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income,reason
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0,invalid_loan_amount;
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0,invalid_action_taken;


## D. Сохранность и повторяемость

In [11]:
again, rejected_again, audit_again = clean_export(export)
assert_frame_equal(result, again)
assert_frame_equal(rejected, rejected_again)
assert audit == audit_again
assert_frame_equal(export, raw_copy)
assert result["source_row"].is_unique
assert result["action_taken"].isin([1, 2, 3]).all()
assert result["loan_amount"].gt(0).all()
assert len(export) == len(result) + len(rejected) + audit["replays"]
print(result["income"].isna().sum())
assert rejected["reason"].ne("").all()

343


## Показатель и выбранная группа

In [12]:
def show_approval(label, table):
    n = len(table)
    assert n > 0
    k = table["action_taken"].isin([1, 2]).sum()
    print(label, k, n, f"{100 * k / n:.2f}%")

show_approval("Оригинал", decisions)
show_approval("После очистки", result)
complete = result.dropna(subset=["income"])
show_approval("Известный доход", complete)

Оригинал 6139 6830 89.88%
После очистки 6137 6828 89.88%
Известный доход 5859 6485 90.35%


## E. Сохранение и отчёт

In [13]:
import json
result.to_csv(WORK / "clean.csv", index=False)
rejected.to_csv(WORK / "quarantine.csv", index=False)
(WORK / "quality_report.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(sorted(p.name for p in WORK.glob("*")))

['clean.csv', 'quality_report.json', 'quarantine.csv']
